<a href="https://colab.research.google.com/github/sangramshitole07/AML/blob/main/AllFeature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dataset loaders
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris

# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Split and cross-validation tools
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score

# Preprocessing and pipeline
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.pipeline import Pipeline

# Filter methods
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif,chi2

# Wrapper methods
from sklearn.feature_selection import SequentialFeatureSelector, RFE, RFECV

# Embedded methods
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report




In [ ]:
data=load_breast_cancer(as_frame=True)
X=data.data
y=data.target
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

Filter method


In [ ]:
#first Varience
varience_filter=VarianceThreshold(threshold=0.3)
X_train_v=varience_filter.fit_transform(X_train)
X_test_v=varience_filter.transform(X_test)
varience_kept=X_train.columns[varience_filter.get_support()]
varience_removed=X_train.columns[~varience_filter.get_support()]
print("original feature count:", X_train.shape[1])
print(f"Varience kept: {varience_kept}")
print(f"Varience removed: {varience_removed}")

original feature count: 30
Varience kept: Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'texture error', 'perimeter error', 'area error', 'worst radius',
       'worst texture', 'worst perimeter', 'worst area'],
      dtype='object')
Varience removed: Index(['mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'smoothness error', 'compactness error',
       'concavity error', 'concave points error', 'symmetry error',
       'fractal dimension error', 'worst smoothness', 'worst compactness',
       'worst concavity', 'worst concave points', 'worst symmetry',
       'worst fractal dimension'],
      dtype='object')


In [ ]:
#2 ) Annova Analysis of variance
K_selector = SelectKBest(score_func=f_classif,k=10)
K_selector.fit(X_train,y_train)

scores=pd.Series(K_selector.scores_,index=X_train.columns).sort_values(ascending=False)
selected_k=X_train.columns[K_selector.get_support()]

print("top score :", scores.nlargest(10))
display(scores.head(10).to_frame("Annova F score "))
print("selected K :",selected_k)


top score : worst concave points    795.223341
worst perimeter         719.598823
mean concave points     703.364031
worst radius            687.430367
mean perimeter          565.485755
worst area              541.713694
mean radius             523.613907
mean area               469.516453
mean concavity          416.374874
worst concavity         365.747389
dtype: float64


,Annova F score
worst concave points,795.223341
worst perimeter,719.598823
mean concave points,703.364031
worst radius,687.430367
mean perimeter,565.485755
worst area,541.713694
mean radius,523.613907
mean area,469.516453
mean concavity,416.374874
worst concavity,365.747389


selected K : Index(['mean radius', 'mean perimeter', 'mean area', 'mean concavity',
       'mean concave points', 'worst radius', 'worst perimeter', 'worst area',
       'worst concavity', 'worst concave points'],
      dtype='object')


In [ ]:
#3 Mutual information
mi_selector=SelectKBest(score_func=mutual_info_classif,k=10)
mi_selector.fit(X_train,y_train)
mi_scores=pd.Series(mi_selector.scores_,index=X_train.columns).sort_values(ascending=False)
print("top MI feature:")

print(mi_scores.head(10))
display(mi_scores.head(15).to_frame("Mutual_Information"))
print("selected :", list(X.columns[mi_selector.get_support()]))

top MI feature:
worst perimeter         0.469783
worst area              0.461707
worst radius            0.457795
mean concave points     0.445031
worst concave points    0.442205
mean perimeter          0.410884
mean radius             0.359714
mean area               0.358809
mean concavity          0.353274
area error              0.336990
dtype: float64


,Mutual_Information
worst perimeter,0.469783
worst area,0.461707
worst radius,0.457795
mean concave points,0.445031
worst concave points,0.442205
mean perimeter,0.410884
mean radius,0.359714
mean area,0.358809
mean concavity,0.353274
area error,0.336990


selected : ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'area error', 'worst radius', 'worst perimeter', 'worst area', 'worst concave points']


In [ ]:
#4 chi square
X_non_negative = MinMaxScaler().fit_transform(X)
selector = SelectKBest(score_func=chi2,k=5)
X_selected = selector.fit_transform(X_non_negative,y)

selected_features = data.feature_names[selector.get_support()]

print("Selected features: ")
print(selected_features)

Selected features: 
['mean concavity' 'mean concave points' 'worst perimeter' 'worst area'
 'worst concave points']


Wrapper Methods

In [ ]:
#1.Forward
